# Qdrant × Future AGI: Fixing An Agentic RAG System That Decayed As It Grew

The agent is broken on purpose and already running against a ~10k-point Qdrant collection. Each section reads three dashboard signals to locate the failing layer: `context_relevance` asks whether the retrieved chunks match the question, `chunk_utilization` asks whether the answer used those chunks, and `groundedness` asks whether the answer stayed inside the retrieved evidence. Then we fix that layer in Qdrant and verify the metric moves on Future AGI.

- **Chat UI** (`app.py`): experience the failure
- **This notebook**: perform the fix and flip the agent's retrieval mode so the chat UI picks it up
- **Future AGI dashboard**: prove it worked

Boundary between the two surfaces: this notebook prints instant gold-label retrieval checks (recall@5, duplicate rate) so a fix's effect is visible the second it lands. The judged quality scores, including the three dashboard signals, the custom answer judge, and the before/after Experiments record, live on the Future AGI dashboard and never appear here.

## 0 · Setup

In [ ]:
import os, json
from pathlib import Path
from dotenv import load_dotenv
from qdrant_client import QdrantClient, models

import agent
from helpers import config, embeddings
from helpers.embeddings import dense, sparse, colbert

load_dotenv()
client = QdrantClient(url=os.environ["QDRANT_URL"], api_key=os.environ["QDRANT_API_KEY"])
COLLECTION = config.COLLECTION

# Start every run from the broken baseline: small dense model, no filter.
agent.set_retrieval(mode="minilm", current_only=False)
embeddings.warmup()   # load all models now, never during a live question
client.count(COLLECTION)

### Tracing (Future AGI): Rishav's Segment
Two lines in `agent.py` (the file on camera) instrument everything:

```python
register(project_name="pokedex-rag")
LangChainInstrumentor().instrument()
```

traceAI instruments the LangGraph agent automatically, so every process that runs it exports the same traces: this notebook, the Streamlit app the audience watches, and the rehearsal scripts. The Qdrant searches show up as retriever spans with no manual span code. Scores are read on the Future AGI dashboard.

## 1 · The Pain: It Decayed as It Grew
Before any fix, score the same golden queries against the collection at 500 / 2k / 10k points. This is a controlled experiment, not a fake six-month timeline: watch retrieval slide as the haystack grows. Recall@5 falls, meaning fewer expected documents make the top-5, while duplicate rate rises, meaning repeated chunks crowd out useful evidence.
(Generated offline by `scaling_curve.py`.)

In [ ]:
from IPython.display import Image
Image("data/scaling_curve.png")   # generated offline by scaling_curve.py

## 2 · Fix #1: Duplicate And Fragmented Chunks → Dedup The Collection
**Dashboard read:** groundedness stays green: the agent honestly works with what it's given. `context_relevance` is also fine: the right topics come back. But the answers are thin, and the retrieval panel shows why: the same `doc_id` fills slot after slot, so few distinct documents ever reach the generator. That pattern points to duplicates, not a weak model.

In [ ]:
# Duplicate rate in the top-k before the fix: a slot is wasted if it repeats a fragment
# with the same doc_id and text we already retrieved.
def dup_rate(query):
    chunks = agent.retrieve(query, mode="minilm")
    keys = [(c["doc_id"], c["text"]) for c in chunks]
    return keys, 1 - len(set(keys)) / len(keys)

keys, rate = dup_rate("Tell me the Pokedex entry for Charizard")
print(f"duplicate rate: {rate:.0%}")
for k in keys: print(" ", k[0])

In [ ]:
# THE FIX: delete duplicate points with the same doc_id + chunk_index, and keep one copy.
from helpers.dedup import find_duplicate_ids

before = client.count(COLLECTION).count
dup_ids = find_duplicate_ids(client)
client.delete(collection_name=COLLECTION, points_selector=dup_ids)
after = client.count(COLLECTION).count
print(f"deleted {len(dup_ids)} duplicates:  {before} -> {after} points")

In [ ]:
keys, rate = dup_rate("Tell me the Pokedex entry for Charizard")
print(f"duplicate rate: {rate:.0%}")   # now distinct documents fill the top-k

**Qdrant Web UI:** open the small `pokemon_viz` collection in the Visualize tab. The duplicate copies sit in tight clusters. Run the same dedup on it and refresh: the clusters visibly thin out. Qdrant also has native MMR (maximal marginal relevance) for query-time diversity, but deduping at the source is the fix.

In [ ]:
# Same dedup on the visualization collection, then refresh the Web UI point cloud.
viz_dups = find_duplicate_ids(client, config.VIZ_COLLECTION)
client.delete(collection_name=config.VIZ_COLLECTION, points_selector=viz_dups)
print(f"pokemon_viz: removed {len(viz_dups)} duplicates")

## 3 · Fix #2: Try A Bigger Embedding Model, Then Measure Before You Commit
**Dashboard read:** after dedup, `context_relevance` is already high: the retriever finds the right topics. The instinct is "upgrade to a bigger embedding model." Qdrant 1.18 lets us add one to the live collection with zero downtime, so this is a safe experiment.

In [ ]:
# Pre-run offline. create_vector_name is instant; backfilling 10k points is the slow part
# and runs in the background. See prep.py. This is the zero-downtime migration:
#   client.create_vector_name(COLLECTION, config.DENSE_STRONG,
#       vector_name_config=models.DenseVectorNameConfig(
#           dense=models.DenseVectorConfig(size=1024, distance=models.Distance.COSINE)))
#   client.update_vectors(COLLECTION, points=[...])   # backfill, payload untouched
# Flip the agent to the candidate model in one line:
agent.set_retrieval(mode="bge")

rows = [json.loads(l) for l in Path("data/golden_dataset.jsonl").read_text().splitlines() if l.strip()]
fix2 = [r for r in rows if r["exercises"] == "fix2_embedding"]

# Instant gold-label check. Judged scores live on the Future AGI dashboard.
def recall(mode):
    hit = 0
    for r in fix2:
        top = {c["doc_id"] for c in agent.retrieve(r["query"], mode=mode)}
        hit += int(bool(top & set(r["gold_doc_ids"])))
    return hit / len(fix2)

print(f"recall@5  small (MiniLM 384d)={recall('minilm'):.2f}   large (bge-large 1024d)={recall('bge'):.2f}")

In [ ]:
# The eval caught a regression: on THIS corpus of short flavor text, the big model is worse.
# Roll back instantly with the same one-line flip. No re-indexing needed.
agent.set_retrieval(mode="minilm")
print("rolled back to:", agent.retrieval_state()["mode"])

The lesson, and the reason this beat exists: **a bigger model is not automatically better on your data.** Public leaderboard rank doesn't transfer; you have to measure. Qdrant made the whole experiment safe: add a named vector to the live collection, A/B test both, and roll back in one line with zero downtime. The retriever already finds the right topics, so the lever was never the model. Next we change the retrieval strategy instead.

## 4 · Fix #3: Dense-Only → Hybrid Retrieval + Late-Interaction Reranking
**Dashboard read:** on paraphrased questions, the right document is not retrieved at all. Nothing downstream can recover that. Groundedness stays green because the agent correctly says "I don't know," so the failure is invisible unless you look at retrieval. Recall@K against the golden set exposes it.

The fix is the recommended production pipeline in one `query_points` call, and we keep MiniLM as the dense side because fix #2 taught us the bigger model is not the lever:

1. **Prefetch** dense + sparse candidates. Dense means embedding similarity; sparse means keyword-style matching that can recover exact words dense loses.
2. **Fuse** the two ranked lists with RRF (reciprocal rank fusion), which rewards documents that rank well in either list.
3. **Rerank** the fused candidates with ColBERT, which compares query and document token by token (late interaction) instead of as two single vectors.

In [ ]:
q = "A Pokemon whose head occasionally drops off and continues on as another Pokemon"

# Dense-only, the current retriever, misses it. The flavor text names the result
# "Exeggcute," which the dense vector never connects to "another Pokemon."
dense_only = [c["doc_id"] for c in agent.retrieve(q, mode="minilm")]
print("dense-only   :", dense_only)

# The hybrid + rerank call. This is what mode='hybrid' runs. See agent.py.
sp = sparse([q], is_query=True)[0]
hybrid = client.query_points(
    COLLECTION,
    prefetch=[models.Prefetch(
        prefetch=[
            models.Prefetch(query=dense([q], config.MODEL_DENSE_WEAK, is_query=True)[0],
                            using=config.DENSE_WEAK, limit=20),
            models.Prefetch(query=models.SparseVector(indices=sp[0], values=sp[1]),
                            using=config.SPARSE, limit=20),
        ],
        query=models.FusionQuery(fusion=models.Fusion.RRF), limit=20,
    )],
    query=colbert([q], is_query=True)[0], using=config.COLBERT, limit=5,
    with_payload=["doc_id"],
).points
print("hybrid+rerank:", [h.payload["doc_id"] for h in hybrid])

In [ ]:
# Flip the agent to hybrid, then score the queries dense retrieval was missing.
agent.set_retrieval(mode="hybrid")

import math
fix3 = [r for r in rows if r["exercises"] == "fix3_hybrid"]

# Instant gold-label check. Judged scores live on the Future AGI dashboard.
def score(mode):
    rec, ndcg, mrr = 0, 0, 0
    for r in fix3:
        seen, ranked = set(), []
        for c in agent.retrieve(r["query"], mode=mode):
            if c["doc_id"] not in seen: seen.add(c["doc_id"]); ranked.append(c["doc_id"])
        ranked, gold = ranked[:5], set(r["gold_doc_ids"])
        rec += int(any(d in gold for d in ranked))
        ndcg += sum(1/math.log2(i+2) for i,d in enumerate(ranked) if d in gold)  # idcg=1
        mrr += next((1/(i+1) for i,d in enumerate(ranked) if d in gold), 0)
    n = len(fix3); return rec/n, ndcg/n, mrr/n

print("recall / NDCG@5 / MRR   dense-only: %.2f / %.2f / %.2f   hybrid: %.2f / %.2f / %.2f"
      % (*score("minilm"), *score("hybrid")))

Two levers, measured separately on the 18 paraphrase queries: **sparse fusion starts the recovery** (recall@5 0.06 → 0.28), and the **ColBERT rerank surfaces the rest of the fused candidates into the top-5** (0.28 → 0.94). This closes on Qdrant's most differentiated capability: the universal Query API, with prefetch, fusion, and late-interaction rerank in one call.

## 5 · Closing The Cold Open: One-Line Payload Filter
Back to the opening failure: the agent said Steel resists Ghost and Dark because a stale pre-Gen-6 type-chart document won retrieval. Here's the uncomfortable part: **dedup and hybrid didn't fix this**. The stale document still ranks first because the question inherits the stale wording ("resists Ghost and Dark" appears verbatim in the outdated chart and not in the current one). Better retrieval can't fix wrong data; only metadata can. Every point carries `is_current` in its payload. Index the field so the filtered search stays fast, then filter on it.

In [ ]:
q = "Does the Steel type resist Ghost and Dark attacks?"
before = [c["doc_id"] for c in agent.retrieve(q, mode="hybrid") if c["name"] == "steel"]
print("no filter :", before)

client.create_payload_index(COLLECTION, "is_current", models.PayloadSchemaType.BOOL)
agent.set_retrieval(current_only=True)
after = [c["doc_id"] for c in agent.retrieve(q, mode="hybrid") if c["name"] == "steel"]
print("is_current:", after)

In [ ]:
# Ask the agent the cold-open question again. Now it is grounded in the current document.
answer, _ = agent.ask("Does the Steel type resist Ghost and Dark attacks?")
print(answer)

## 6 · Bonus: Multi-Hop In The Trace View (Future AGI)
A compound question makes the agent retrieve twice. Watch it split the task into two retriever spans in the Future AGI trace. That is the proof this is an agent, not a single-shot retriever.

In [ ]:
answer, chunks = agent.ask("What does Drowzee eat, and is that Pokemon weak to Bug-type attacks?")
print(answer)